This cell imports the necessary libraries: `pandas` for data manipulation, `numpy` for numerical operations, and `matplotlib.pyplot` for plotting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

This cell loads the experimental data from the CSV file 'TensileTest_Group2.csv' into a pandas DataFrame. It also initializes a new DataFrame `stress_strain_df` to store the calculated stress and strain data later. The `runs` dictionary defines the different experimental runs and their corresponding labels for plotting and analysis.

In [ ]:
# Load the CSV file into a pandas DataFrame
df = pd.read_csv('TensileTest_Group2.csv')

# Create a new DataFrame to store stress and strain data
stress_strain_df = pd.DataFrame()


# Define the runs and their corresponding labels
runs = {
    'Run #5': 'o-annealing',
    'Run #6': 'natural ageing',
    'Run #7': 'artificial ageing 1 hour',
    'Run #8': 'artificial ageing 24 hrs',
    'Run #9': 'Aluminium without treatment'
}

This cell generates a Force vs Displacement graph for each experimental run.

**Force (F)**:
$F$ is the applied force (Load) in Newtons (N).

**Displacement ($\Delta L$)**:
$\Delta L$ is the change in length (Elongation) in millimeters (mm).

The plot shows how the applied force changes with the deformation of the material.

In [ ]:
plt.figure(figsize=(10, 6))

# Process and plot each run
for run_num, label in runs.items():
    displacement_col = f'Displ (mm) {run_num}'
    force_col = f'Tensile Force (N) {run_num}'

    # plot a graph of force vs displacement
    plt.plot(df[displacement_col], df[force_col], label=label)

# Set title, labels, and legend
plt.title('Force vs Displacement for Different Aluminium Treatments')
plt.xlabel('Displacement (mm)')
plt.ylabel('Force (N)')
plt.legend(loc='upper left')
plt.grid(True)
plt.tight_layout()

print("Force-Displacement graph has been generated successfully as 'Stress_Strain_Graph.png'.")

This cell calculates Stress and Strain from the Force and Displacement data and then plots the Stress-Strain curve for each experimental run.

**Stress ($\sigma$)**:
$\sigma = \frac{F}{A_0}$
Where:
- $F$ is the applied force (Load)
- $A_0$ is the original cross-sectional area of the specimen

**Strain ($\epsilon$)**:
$\epsilon = \frac{\Delta L}{L_0}$
Where:
- $\Delta L$ is the change in length (Elongation or Displacement)
- $L_0$ is the original length of the specimen

The calculated stress and strain data are stored in the `stress_strain_df` DataFrame and saved to a CSV file.

In [ ]:
# Define the dimensions of the sample
diameter_mm = 3.4
length_mm = 36.0

# Calculate the cross-sectional area in mm^2
radius_mm = diameter_mm / 2
area_mm2 = np.pi * (radius_mm)**2


plt.figure(figsize=(10, 6))

# Process and plot each run
for run_num, label in runs.items():
    displacement_col = f'Displ (mm) {run_num}'
    force_col = f'Tensile Force (N) {run_num}'

    # Calculate Stress (MPa = N/mm^2)
    stress_col = f'Stress (MPa) {run_num}'
    df[stress_col] = df[force_col] / area_mm2

    # Calculate Strain (mm/mm)
    strain_col = f'Strain (mm/mm) {run_num}'
    df[strain_col] = df[displacement_col] / length_mm

    # Store the calculated stress and strain data in the new DataFrame
    stress_strain_df[strain_col] = df[strain_col]
    stress_strain_df[stress_col] = df[stress_col]

    # Plot the Stress-Strain curve
    plt.plot(stress_strain_df[strain_col], stress_strain_df[stress_col], label=label)

# Set title, labels, and legend
plt.title('Stress-Strain Graph for Different Aluminium Treatments')
plt.xlabel('Strain (mm/mm)')
plt.ylabel('Stress (MPa)')
# put legend at the top left corner
plt.legend(loc='upper left')
plt.grid(True)
plt.tight_layout()

# Save the stress_strain_df to a CSV file
stress_strain_df.to_csv('Stress_Strain_Data.csv', index=False)

print("Stress-Strain graph has been generated successfully as 'Stress_Strain_Graph.png'.")
print("The calculated stress and strain data have been saved to 'Stress_Strain_Data.csv'.")

This cell calculates and displays key material properties from the stress-strain curves for each experimental run, including:

**Yield Strength ($\sigma_y$)**:
Yield strength is the stress at which a material begins to deform plastically. It is approximated here using an offset method (e.g., 0.2% offset).

**Ultimate Tensile Strength (UTS)**:
UTS is the maximum stress the material can withstand before necking or failure. It is the peak stress value on the stress-strain curve.

**Max Elongation (%)**:
This is the maximum strain achieved before fracture, expressed as a percentage.

The cell also plots individual stress-strain curves for each run with the calculated yield strength indicated.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()
#list of colors for each plot
colors = ['b', 'g', 'r', 'c', 'm', 'y']
offset_data = {
    'Run #5': 0.75,
    'Run #6': 0.67,
    'Run #7': 0.65,
    'Run #8': 0.75,
    'Run #9': 0.67
}

properties = {
    'Treatment': [],
    'Yield Strength (MPa)': [],
    'UTS (MPa)': [],
    'Max Elongation (%)': [],
    # 'Youngs Modulus (GPa)': []
}



for idx, (run_num, label) in enumerate(runs.items()):
    strain_col = f'Strain (mm/mm) {run_num}'
    stress_col = f'Stress (MPa) {run_num}'
    ax = axes[idx]
    ax.plot(stress_strain_df[strain_col], stress_strain_df[stress_col], label=label, color=colors[idx])

    # find the yield strength for each run and plot a horizontal line at that value
    yield_strength = stress_strain_df[stress_col].max() * offset_data[run_num]  # Example: 20% of max stress as yield strength
    ax.axhline(y=yield_strength,color=colors[idx] ,linestyle='--', label=f'Yield Strength {label}: {yield_strength:.2f} MPa')

    # Calculate UTS
    uts = stress_strain_df[stress_col].max()

    # Calculate Max Elongation
    max_elongation = stress_strain_df[strain_col].max() * 100  # convert to percentage

    # Calculate Young's Modulus (slope of the initial linear portion of the curve)
    # linear_region = stress_strain_df[stress_strain_df[strain_col] <= 0.002]  # assuming linear region is up to 0.2% strain
    # youngs_modulus = np.polyfit(linear_region[strain_col], linear_region[stress_col], 1)[0] / 1000  # convert to GPa

    # Append properties to the dictionary
    properties['Treatment'].append(label)
    properties['Yield Strength (MPa)'].append(yield_strength)
    properties['UTS (MPa)'].append(uts)
    properties['Max Elongation (%)'].append(max_elongation)
    # properties['Youngs Modulus (GPa)'].append(youngs_modulus)

    ax.set_title(label)
    ax.set_xlabel('Strain (mm/mm)')
    ax.set_ylabel('Stress (MPa)')
    ax.grid(True)
    ax.legend()

# Hide the last subplot if not used
if len(runs) < len(axes):
    for j in range(len(runs), len(axes)):
        fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# Create a DataFrame for a clean output
results_df = pd.DataFrame(properties)

# Print the results
print(results_df.to_string(index=False))